# Prospect Client Analysis — Google Colab
This notebook uses the no-API-key Playwright Google Maps browser collector, then runs the same ETL, enrichment, ML, analytics, and export pipeline as the Streamlit dashboard.


In [ ]:
!git clone https://github.com/softmitend/prospect-client-analysis.git
%cd prospect-client-analysis
!pip -q install -r requirements.txt
!python -m playwright install --with-deps chromium


In [ ]:
LOCATION = 'Banyuwangi, East Java, Indonesia'
CATEGORIES = 'restaurant,cafe,hotel,dentist,clinic,beauty salon,lawyer,car repair'
MAX_RESULTS = 20


In [ ]:
import subprocess
cmd = [
    'python', 'bin/run_pipeline.py',
    '--provider', 'playwright',
    '--browser-channel', '',
    '--location', LOCATION,
    '--categories', CATEGORIES,
    '--max-results-per-category', str(MAX_RESULTS),
    '--min-category-size', '3',
    '--enrich-websites',
    '--output', 'data/colab_results.xlsx',
]
subprocess.run(cmd, check=True)


In [ ]:
import pandas as pd
businesses = pd.read_excel('data/colab_results.xlsx', sheet_name='businesses')
summary = pd.read_excel('data/colab_results.xlsx', sheet_name='category_summary')
summary


In [ ]:
cols = [c for c in ['business_name','category','website','public_email','whatsapp_number','phone','address','opening_hours_text','digital_presence_score','ml_cluster','ml_anomaly'] if c in businesses]
businesses[cols].head(30)


In [ ]:
import matplotlib.pyplot as plt
plot_df = summary.sort_values('website_adoption_pct')
plt.figure(figsize=(10, max(4, len(plot_df)*0.35)))
plt.barh(plot_df['category'], plot_df['website_adoption_pct'])
plt.xlabel('Website adoption (%)')
plt.title('Website adoption by business category')
plt.tight_layout()
plt.show()


## Notes
The browser collector is intentionally conservative. It does not bypass CAPTCHA or anti-bot controls. Google Maps UI selectors may change, so a browser-scraping collector requires maintenance. `ml_cluster` uses TF-IDF + MiniBatchKMeans and `ml_anomaly` uses Isolation Forest; website-adoption percentages remain deterministic aggregations.
